In [2]:
import json
import os
import openai

def get_current_temperature(location: str, unit: str = "celsius"):
    """Get current temperature at a location.

    Args:
        location: The location to get the temperature for, in the format "City".
        unit: The unit to return the temperature in. Defaults to "celsius". (choices: ["celsius", "fahrenheit"])

    Returns:
        the temperature, the location, and the unit in a dict
    """
    return {
        "temperature": 24,
        "location": location,
        "unit": unit,
    }


def get_temperature_date(location: str, date: str, unit: str = "celsius"):
    """Get temperature at a location and date.

    Args:
        location: The location to get the temperature for, in the format "City, State, Country".
        date: The date to get the temperature for, in the format "Year-Month-Day".
        unit: The unit to return the temperature in. Defaults to "celsius". (choices: ["celsius", "fahrenheit"])

    Returns:
        the temperature, the location, the date and the unit in a dict
    """
    return {
        "temperature": 32,
        "location": location,
        "date": date,
        "unit": unit,
    }




In [3]:
def get_function_by_name(name):
    if name == "get_current_temperature":
        return get_current_temperature
    if name == "get_temperature_date":
        return get_temperature_date

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_temperature",
            "description": "Get current temperature at a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": 'The location to get the temperature for, in the format "City, State, Country".',
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": 'The unit to return the temperature in. Defaults to "celsius".',
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_temperature_date",
            "description": "Get temperature at a location and date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": 'The location to get the temperature for, in the format "City, State, Country".',
                    },
                    "date": {
                        "type": "string",
                        "description": 'The date to get the temperature for, in the format "Year-Month-Day".',
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": 'The unit to return the temperature in. Defaults to "celsius".',
                    },
                },
                "required": ["location", "date"],
            },
        },
    },
]



LLM根据用户的输入，自动选择合适的tool(工具、函数)，获得相应的结果。

In [4]:
MESSAGES = [
    {"role": "system", "content": "You are a helpful assistant.\n\n"},
    {"role": "user",  "content": "沈阳今天天气怎么样?? "},   #"大连今天是2025-04-17日，今天的天气怎么样??"
]


MODEL_NAME="qwen-plus"
#
tools = TOOLS
messages = MESSAGES[:]

client = openai.OpenAI(
            api_key="sk-b0a9299097b04f7baef9f3254fc273e9",
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        )

response1 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages)

print("===1. LLM Input===\n")
print(messages)
print("===1. LLM Input End===\n")

print("===1. LLM Output(Function Call)===\n")
print(response1.choices[0])
print("===1. LLM Output(Function Call) End===\n")

response2 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

output_text2 = response2.choices[0].message

print("===2. LLM Output===\n")

print("XXXXX:",output_text2)

print("===2. LLM Output End===\n")
 # 检查是否有普通文本内容
if output_text2.content:
    print(output_text2.content)
    process_query = False
    
# 检查是否有工具调用
elif output_text2.tool_calls:
    # 添加助手消息到历史
    messages.append({
        "role": "assistant", 
        "content": None,
        "tool_calls": output_text2.tool_calls
    })

    for tool_call in output_text2.tool_calls:
        call_id = tool_call.id
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        print(f"Calling tool {tool_name} with args {tool_args}")
        fn_res = json.dumps(get_function_by_name(tool_name)(**tool_args))

        messages.append({
            "role": "tool",
            "content": fn_res,
            "tool_call_id": call_id,
        })

print("===3. LLM Input===\n")
print(messages)

print("===3. LLM Input End===\n")

chat_completion_3 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

output_text3 = chat_completion_3.choices[0].message.content

print("===3. LLM Output===\n")

print("XXXXX:",chat_completion_3.choices[0].message)

#print(output_text3)
print("===3. LLM Output End===\n")


===1. LLM Input===

[{'role': 'system', 'content': 'You are a helpful assistant.\n\n'}, {'role': 'user', 'content': '沈阳今天天气怎么样?? '}]
===1. LLM Input End===

===1. LLM Output(Function Call)===

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='我无法获取实时天气信息。建议您通过天气预报网站（如中国天气网）或手机天气应用查看沈阳的最新天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))
===1. LLM Output(Function Call) End===

===2. LLM Output===

XXXXX: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_b9ef651f51d74b69ae7e9a', function=Function(arguments='{"location": "Shenyang, Liaoning, China"}', name='get_current_temperature'), type='function', index=0)])
===2. LLM Output End===

Calling tool get_current_temperature with args {'location': 'Shenyang, Liaoning, China'}
===3. LLM Input===

[{'role': 'system', 'conte